# UI-Flow Haptic Pipeline v1

This notebook is the first pipeline step after the behavior atlas. It does **not** call an LLM yet. Instead, it takes a manual `HAPTIC_PLAN`, selects one curated raw-z prototype, decodes one 0.5s VAE chunk, and inserts that chunk into a 2s silent waveform at the beginning, middle, or end.

In [ ]:
# -----------------------------
# Editable notebook parameters
# -----------------------------
from pathlib import Path

CONFIG_PATH = "configs/vae_balanced_8d_0p5s.yaml"
CHECKPOINT_PATH_OVERRIDE = None  # Example: "/content/drive/MyDrive/thesis_outputs/outputs/.../best_model.pt"

DRIVE_ROOT = Path("/content/drive/MyDrive/thesis_outputs")
OUTPUTS_ROOT = DRIVE_ROOT / "outputs"
ATLAS_DIR = DRIVE_ROOT / "behavior_atlas_8class"
CURATED_JSON_PATH = ATLAS_DIR / "curated_behavior_prototypes.json"
ATLAS_JSON_PATH = ATLAS_DIR / "behavior_atlas.json"
PIPELINE_OUTPUT_DIR = DRIVE_ROOT / "ui_flow_haptic_v1"

# v1 supports exactly one behavior per run.
# placement: "beginning", "middle", or "end"
HAPTIC_PLAN = {
    "interaction_type": "slider_drag",
    "behavior": {
        "class": "long_rising",
        "strength": 0.7,
        "placement": "middle",
        "prototype_policy": "curated_first",
        "prototype_index": 0,
    },
}

GENERATED_BASENAME = "generated_haptic"
FINAL_DURATION_S = 2.0
FADE_MS = 10.0

## Future LLM Contract

Later, a multimodal LLM should read `before`, `during`, and `after` UI screenshots and output the same plan shape used above. It should choose exactly one behavior class for v1.

Allowed classes:
- `quiet_or_subtle`
- `single_short_pulse`
- `double_short_pulse`
- `multi_short_pulse`
- `long_sustained`
- `long_rising`
- `long_falling`
- `long_modulated_or_pulsed`

Allowed placements:
- `beginning`
- `middle`
- `end`

For now, edit `HAPTIC_PLAN` manually and run the notebook top to bottom.

In [ ]:
# -----------------------------
# Setup, Drive mount, imports
# -----------------------------
import json
import os
import sys
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Audio, display
from scipy.io.wavfile import write as wav_write

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Google Drive mount skipped or unavailable: {exc}")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src").exists():
            PROJECT_ROOT = parent
            break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loaders import build_model, load_checkpoint
from src.utils.config import load_config

PIPELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Pipeline output dir: {PIPELINE_OUTPUT_DIR}")

In [ ]:
# -----------------------------
# File resolution and atlas helpers
# -----------------------------
def resolve_checkpoint(outputs_root: Path, override: str | None = None) -> Path:
    if override:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(f"CHECKPOINT_PATH_OVERRIDE does not exist: {path}")
        print(f"Using override checkpoint: {path}")
        return path

    if not outputs_root.exists():
        raise FileNotFoundError(f"OUTPUTS_ROOT does not exist: {outputs_root}")

    candidates = []
    for name in ("best_model.pt", "last_checkpoint.pt"):
        candidates.extend(outputs_root.rglob(name))
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No best_model.pt or last_checkpoint.pt found under: {outputs_root}")

    print("Checkpoint candidates, newest first:")
    for idx, path in enumerate(candidates[:10]):
        print(f"  [{idx}] {path}")
    print(f"Selected checkpoint: {candidates[0]}")
    return candidates[0]


def load_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_prototype_source(curated_path: Path, atlas_path: Path) -> tuple[dict | None, dict | None]:
    curated = load_json(curated_path) if curated_path.exists() else None
    atlas = load_json(atlas_path) if atlas_path.exists() else None
    if curated is None and atlas is None:
        raise FileNotFoundError(
            f"Missing both curated prototype file and atlas file:\n  {curated_path}\n  {atlas_path}"
        )
    if curated is not None:
        print(f"Loaded curated prototypes: {curated_path}")
    else:
        print(f"Curated prototypes not found, using atlas only: {atlas_path}")
    if atlas is not None:
        print(f"Loaded atlas: {atlas_path}")
    return curated, atlas


def prototype_from_class(
    behavior_class: str,
    curated: dict | None,
    atlas: dict | None,
    prototype_index: int = 0,
) -> dict | None:
    if curated is not None:
        class_entry = curated.get("classes", {}).get(behavior_class, {})
        approved = class_entry.get("approved_prototypes", []) or []
        if approved:
            idx = min(max(int(prototype_index), 0), len(approved) - 1)
            proto = dict(approved[idx])
            proto["selection_source"] = "curated.approved_prototypes"
            proto["selection_index"] = idx
            return proto
        fallback = class_entry.get("fallback_prototype")
        if fallback:
            proto = dict(fallback)
            proto["selection_source"] = "curated.fallback_prototype"
            proto["selection_index"] = 0
            return proto

    if atlas is not None:
        prototypes = atlas.get("classes", {}).get(behavior_class, {}).get("prototype_z", []) or []
        if prototypes:
            idx = min(max(int(prototype_index), 0), len(prototypes) - 1)
            proto = dict(prototypes[idx])
            proto["selection_source"] = "atlas.prototype_z"
            proto["selection_index"] = idx
            return proto

    return None


def validate_plan(plan: dict) -> dict:
    behavior = dict(plan.get("behavior", {}))
    if "class" not in behavior:
        raise ValueError("HAPTIC_PLAN['behavior']['class'] is required")
    behavior["strength"] = float(np.clip(float(behavior.get("strength", 0.7)), 0.0, 1.0))
    behavior["placement"] = str(behavior.get("placement", "middle"))
    if behavior["placement"] not in {"beginning", "middle", "end"}:
        raise ValueError("placement must be one of: beginning, middle, end")
    behavior["prototype_index"] = int(behavior.get("prototype_index", 0))
    return behavior

In [ ]:
# -----------------------------
# Load VAE, checkpoint, and atlas files
# -----------------------------
config = load_config(str(PROJECT_ROOT / CONFIG_PATH))
sr = int(config["data"]["sr"])
chunk_len = int(config["data"]["T"])
latent_dim = int(config["model"]["latent_dim"])
final_len = int(round(FINAL_DURATION_S * sr))
clip_range = config.get("loss", {}).get("clamp_range", None)
data_clip_range = config.get("data", {}).get("clip_range", None)
if clip_range is None and data_clip_range is not None:
    clip_abs = max(abs(float(data_clip_range[0])), abs(float(data_clip_range[1])))
else:
    clip_abs = float(clip_range or 5.0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = resolve_checkpoint(OUTPUTS_ROOT, CHECKPOINT_PATH_OVERRIDE)
curated_data, atlas_data = load_prototype_source(CURATED_JSON_PATH, ATLAS_JSON_PATH)

model = build_model(config, device)
load_checkpoint(model, str(checkpoint_path), device)
model.eval()

print(f"Device: {device}")
print(f"sr={sr}, chunk_len={chunk_len}, final_len={final_len}, latent_dim={latent_dim}")
print(f"clip_abs={clip_abs}")

In [ ]:
# -----------------------------
# Decode one behavior and insert it into 2s silence
# -----------------------------
def decode_z(z_values: list[float] | np.ndarray) -> np.ndarray:
    z = np.asarray(z_values, dtype=np.float32).reshape(1, -1)
    if z.shape[1] != latent_dim:
        raise ValueError(f"Expected latent_dim={latent_dim}, got z shape {z.shape}")
    z_t = torch.from_numpy(z).to(device)
    with torch.no_grad():
        decoded = model.decode(z_t, target_len=chunk_len)
    return decoded[0, 0].detach().cpu().numpy().astype(np.float32)


def fade_edges(x: np.ndarray, sr: int, fade_ms: float) -> np.ndarray:
    y = np.asarray(x, dtype=np.float32).copy()
    fade_n = int(round(sr * fade_ms / 1000.0))
    fade_n = max(0, min(fade_n, len(y) // 2))
    if fade_n > 1:
        ramp = np.linspace(0.0, 1.0, fade_n, dtype=np.float32)
        y[:fade_n] *= ramp
        y[-fade_n:] *= ramp[::-1]
    return y


def placement_start_index(placement: str, final_len: int, chunk_len: int) -> int:
    if placement == "beginning":
        return 0
    if placement == "middle":
        return (final_len - chunk_len) // 2
    if placement == "end":
        return final_len - chunk_len
    raise ValueError(f"Unsupported placement: {placement}")


behavior = validate_plan(HAPTIC_PLAN)
behavior_class = behavior["class"]
strength = behavior["strength"]
placement = behavior["placement"]
prototype_index = behavior["prototype_index"]

selected_proto = prototype_from_class(
    behavior_class,
    curated=curated_data,
    atlas=atlas_data,
    prototype_index=prototype_index,
)

final_wave = np.zeros(final_len, dtype=np.float32)
if selected_proto is None:
    decoded_chunk = np.zeros(chunk_len, dtype=np.float32)
    selection_summary = {
        "behavior_class": behavior_class,
        "selection_source": "none_silence",
        "sample_index": None,
        "file_path": None,
    }
else:
    decoded_chunk = decode_z(selected_proto["z"])
    gain = 0.5 + 0.8 * strength
    decoded_chunk = decoded_chunk * gain
    decoded_chunk = fade_edges(decoded_chunk, sr=sr, fade_ms=FADE_MS)
    decoded_chunk = np.clip(decoded_chunk, -clip_abs, clip_abs).astype(np.float32)
    selection_summary = {
        "behavior_class": behavior_class,
        "selection_source": selected_proto.get("selection_source"),
        "selection_index": selected_proto.get("selection_index"),
        "sample_index": selected_proto.get("sample_index"),
        "file_path": selected_proto.get("file_path"),
        "atlas_class": selected_proto.get("atlas_class"),
        "curated_class": selected_proto.get("curated_class"),
        "gain": gain,
    }

start_idx = placement_start_index(placement, final_len=final_len, chunk_len=chunk_len)
end_idx = start_idx + chunk_len
final_wave[start_idx:end_idx] = decoded_chunk
final_wave = np.clip(final_wave, -clip_abs, clip_abs).astype(np.float32)

print("Selected behavior:")
print(json.dumps({"plan": HAPTIC_PLAN, "selection": selection_summary}, indent=2))
print(f"placement={placement}, start_idx={start_idx}, end_idx={end_idx}")
print(f"chunk max={np.max(np.abs(decoded_chunk)):.4f}, final max={np.max(np.abs(final_wave)):.4f}")

In [ ]:
# -----------------------------
# Save generated files
# -----------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = PIPELINE_OUTPUT_DIR / f"{GENERATED_BASENAME}_{behavior_class}_{placement}_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=True)

npy_path = run_dir / f"{GENERATED_BASENAME}.npy"
wav_path = run_dir / f"{GENERATED_BASENAME}.wav"
plan_path = run_dir / "haptic_plan.json"
preview_path = run_dir / "waveform_preview.png"

np.save(npy_path, final_wave)
wav_write(wav_path, sr, final_wave.astype(np.float32))

saved_plan = {
    "haptic_plan": HAPTIC_PLAN,
    "selection": selection_summary,
    "sr": sr,
    "final_duration_s": FINAL_DURATION_S,
    "final_len": final_len,
    "chunk_len": chunk_len,
    "placement": placement,
    "start_idx": start_idx,
    "end_idx": end_idx,
    "checkpoint_path": str(checkpoint_path),
    "curated_json_path": str(CURATED_JSON_PATH),
    "atlas_json_path": str(ATLAS_JSON_PATH),
}
with open(plan_path, "w", encoding="utf-8") as f:
    json.dump(saved_plan, f, indent=2)

print(f"Saved output directory: {run_dir}")
print(f"WAV:  {wav_path}")
print(f"NPY:  {npy_path}")
print(f"PLAN: {plan_path}")

In [ ]:
# -----------------------------
# Plot and playback
# -----------------------------
time_final = np.arange(final_len) / sr
time_chunk = np.arange(chunk_len) / sr

fig, axes = plt.subplots(2, 1, figsize=(15, 6), gridspec_kw={"height_ratios": [1, 2]})
axes[0].plot(time_chunk, decoded_chunk, linewidth=1.0)
axes[0].set_title("Decoded 0.5s chunk")
axes[0].set_xlabel("time within chunk (s)")
axes[0].set_ylabel("amplitude")
axes[0].grid(alpha=0.2)

axes[1].plot(time_final, final_wave, linewidth=1.0, label="2s output")
axes[1].axvspan(start_idx / sr, end_idx / sr, color="orange", alpha=0.18, label="inserted chunk")
axes[1].set_title(f"Final 2s waveform | class={behavior_class}, placement={placement}")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("amplitude")
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig(preview_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved preview: {preview_path}")
display(Audio(final_wave, rate=sr))

In [ ]:
# -----------------------------
# Smoke assertions for v1 behavior
# -----------------------------
expected_starts = {
    "beginning": 0,
    "middle": (final_len - chunk_len) // 2,
    "end": final_len - chunk_len,
}
assert len(final_wave) == 16000, f"Expected 16000 samples for 2s at 8kHz, got {len(final_wave)}"
assert chunk_len == 4000, f"Expected 4000-sample VAE chunk, got {chunk_len}"
assert start_idx == expected_starts[placement], (placement, start_idx, expected_starts[placement])

outside = final_wave.copy()
outside[start_idx:end_idx] = 0.0
assert np.max(np.abs(outside)) < 1e-7, "Samples outside the inserted 0.5s region should remain silent"

print("Smoke checks passed.")
print(f"beginning start index: {expected_starts['beginning']}")
print(f"middle start index:    {expected_starts['middle']}")
print(f"end start index:       {expected_starts['end']}")